Importing drawing library

In [40]:
import matplotlib
import matplotlib.pyplot as plt

Importing utility

In [ ]:
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence
from util import *

Qiskit circuit definition

In [42]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz

Qiskit Backend

In [43]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

Qiskit hight performance simulation

In [44]:
from qiskit_aer import AerSimulator
from qiskit.transpiler import generate_preset_pass_manager

Manipulate QUBO problem

In [45]:
import qubovert

In [ ]:
restarting = 15
batch = 30

Generate QUBO Problem

In [ ]:
def gen_qubo(verilog_source):
    try:
        path = os.getcwd()
        subprocess.run(['yosys', verilog_source, 'work/synth.ys', '-b', 'edif', '-o', 'work/circ_sat.edif'], capture_output=True)
        subprocess.run(['edif2qmasm', '-o', 'work/circ_sat.qmasm', 'work/circ_sat.edif'])
        os.chdir('work')
        subprocess.run(['qmasm', '--format', 'numpy', '-o', '../sat.npz', '--solver', 'tabu', '--pin', 'circ_sat.y := true', 'circ_sat.qmasm'])
        os.chdir(path)
        return 0
    except:
        return 1

['circ_sat.$id00005.A' 'circ_sat.$id00005.B' 'circ_sat.$id00005.Y'
 'circ_sat.$id00006.A' 'circ_sat.$id00006.B' 'circ_sat.$id00006.Y'
 'circ_sat.$id00004.A' 'circ_sat.$id00004.Y' 'circ_sat.b' 'circ_sat.a'
 'circ_sat.c']


In [ ]:
def gen_ising(qubo_mat):
    ising_dict, ising_mat = convert_to_ising(qubo_mat)
    ising_solution = qubovert.QUSO(ising_dict).solve_bruteforce()
    ising_energy = qubovert.QUSO(ising_dict).value(ising_solution)
    return ising_mat, ising_energy

In [ ]:
def initialize_tsv(file):
    column = ["method", "rep", "energy", "n_eval", "runtime", "000", "001", "010", "011", "100", "101", "110", "111"]
    with open(file, 'w') as out:
        for col in column:
            out.write(col + "\t")
        out.write('\n')

In [ ]:
def write_tsv(res, file):
    with open(file, 'a') as out:
        for freq in res:
            out.write('\t' + str(freq[1]))
        out.write('\n')
    return 0

In [ ]:
def run_experiments(ansatz, backend, candidate_circuit, cost_hamiltonian, restarting, opt_method, file_tsv):
    for i in range(batch):
        energy, n_eval, res, run_time = run_experiment(ansatz, backend, candidate_circuit, cost_hamiltonian, restarting, opt_method)
        print(opt_method, i, run_time, sep='\t')
        write_tsv(res, file_tsv)

In [ ]:
def __main__():
    error = gen_qubo('circ_sat.v')
    data, qubo_mat, n_qubits = load_qubo('sat.npz')
    ising_mat, ising_energy = gen_ising(qubo_mat)
    cost_hamiltonian = SparsePauliOp.from_sparse_list(build_paulis(ising_mat), n_qubits)
    ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=5)
    ansatz.measure_all()
    backend = AerSimulator()
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    candidate_circuit = pm.run(ansatz)

    initialize_tsv('raw_data.tsv')
    run_experiments(ansatz, backend, candidate_circuit, cost_hamiltonian, restarting, 'COBYLA', 'raw_data.tsv')
    return 0

In [ ]:
__main__()